# Notebook 08 — Hybrid Candidate Generation (Popularity + Item-CF + FAISS)

**Goal:** combine candidate sources with deterministic Reciprocal Rank Fusion (RRF) and compare against the strongest single-source baseline using the *same* warm-user, future-view, exclude-seen evaluation protocol.

**Important:** Notebook 07's exported FAISS candidates already contain popularity fallback, so that source is labeled **FAISS+fallback**, not pure vector candidates. Notebook 08 rebuilds Item-CF from historical data using the same settings as Notebook 05. No future labels are used for feature generation. Validation is for model choice; test is for final reporting only. This is an offline replay study, not unbiased CTR measurement.

## 1. Load and validate input files

In [1]:
from pathlib import Path
from collections import defaultdict
import json, time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

cwd = Path.cwd().resolve()
if (cwd/'data'/'processed').is_dir(): ROOT=cwd
elif (cwd.parent/'data'/'processed').is_dir(): ROOT=cwd.parent
else: raise FileNotFoundError(f'Cannot find data/processed from {cwd}')
PROCESSED=ROOT/'data'/'processed'
REPORTS=ROOT/'reports'; REPORTS.mkdir(parents=True,exist_ok=True)
CUTOFFS={'val':pd.Timestamp('2015-08-01',tz='UTC'), 'test':pd.Timestamp('2015-09-01',tz='UTC')}
K_VALUES=[10,20,50,100]; MAX_K=100
# Fixed before looking at hybrid test results; can tune using validation only.
RRF_C=60.0
SOURCE_WEIGHTS={'itemcf':1.0,'faiss_fallback':1.0,'popularity':0.3}
inputs={}
for split,stem in [('val','validation'),('test','test')]:
    history=pd.read_csv(PROCESSED/('train_history.csv' if split=='val' else 'test_history.csv'))
    labels=pd.read_csv(PROCESSED/f'{stem}_view_labels_warm_users.csv')
    fp=PROCESSED/f'notebook07_{split}_faiss_candidates.csv'
    assert fp.exists(),f'Missing {fp}; run Notebook 07 first'
    faiss_rows=pd.read_csv(fp)
    assert history.timestamp.max()<int(CUTOFFS[split].timestamp()*1000)
    assert labels.visitorid.isin(history.visitorid).all()
    assert not labels[['visitorid','itemid']].duplicated().any()
    assert not faiss_rows[['visitorid','rank']].duplicated().any()
    assert not faiss_rows[['visitorid','itemid']].duplicated().any()
    users=sorted(map(int,labels.visitorid.unique()))
    assert set(map(int,faiss_rows.visitorid.unique()))==set(users)
    assert faiss_rows.groupby('visitorid').size().eq(MAX_K).all()
    inputs[split]={'history':history,'labels':labels,'faiss_rows':faiss_rows,'users':users}
    print(split,'history',len(history),'users',len(users),'labels',len(labels),'FAISS rows',len(faiss_rows))
print('PASS: input and temporal integrity')

val history 1899029 users 13329 labels 28634 FAISS rows 1332900
test history 2452391 users 15137 labels 31876 FAISS rows 1513700
PASS: input and temporal integrity


## 2. Rebuild Item-CF using Notebook 05 settings (history only)

In [2]:
MAX_ITEMS_PER_USER=20
MAX_ITEM_USER_FREQUENCY=2000
TOP_NEIGHBORS=50
SEED_ITEMS=20

def fit_itemcf(history):
    views=history.loc[history.event.eq('view'),['visitorid','itemid','timestamp']]
    views=views.sort_values('timestamp',ascending=False,kind='mergesort')
    views=views.drop_duplicates(['visitorid','itemid'],keep='first')
    views=views.groupby('visitorid',sort=False).head(MAX_ITEMS_PER_USER)
    freq=views.itemid.value_counts()
    views=views.loc[views.itemid.isin(freq.index[freq<=MAX_ITEM_USER_FREQUENCY]),['visitorid','itemid']]
    user_ids,user_idx=np.unique(views.visitorid.to_numpy(),return_inverse=True)
    item_ids,item_idx=np.unique(views.itemid.to_numpy(),return_inverse=True)
    X=csr_matrix((np.ones(len(views),dtype=np.float32),(user_idx,item_idx)),shape=(len(user_ids),len(item_ids)))
    X.sum_duplicates()
    cooc=(X.T@X).tocsr();cooc.setdiag(0);cooc.eliminate_zeros()
    degrees=np.asarray(X.sum(axis=0)).ravel()
    neighbors={}
    for i in range(cooc.shape[0]):
        start,end=cooc.indptr[i],cooc.indptr[i+1]
        indices=cooc.indices[start:end]
        if not len(indices):continue
        scores=cooc.data[start:end]/np.sqrt(degrees[i]*degrees[indices])
        if len(indices)>TOP_NEIGHBORS:
            keep=np.argpartition(scores,-TOP_NEIGHBORS)[-TOP_NEIGHBORS:]
            indices,scores=indices[keep],scores[keep]
        order=np.lexsort((item_ids[indices],-scores))
        neighbors[int(item_ids[i])]=[(int(item_ids[indices[j]]),float(scores[j])) for j in order]
    return neighbors

def prepare_users(history,users):
    sub=history.loc[history.visitorid.isin(users),['visitorid','itemid','event','timestamp']]
    seen=sub.groupby('visitorid').itemid.agg(lambda x:set(map(int,x))).to_dict()
    views=sub.loc[sub.event.eq('view')].sort_values('timestamp',ascending=False,kind='mergesort')
    views=views.drop_duplicates(['visitorid','itemid'],keep='first')
    seeds=views.groupby('visitorid',sort=False).itemid.agg(lambda x:list(map(int,x.head(SEED_ITEMS)))).to_dict()
    counts=history.loc[history.event.eq('view'),'itemid'].value_counts()
    pop=(counts.rename_axis('itemid').reset_index(name='count')
         .sort_values(['count','itemid'],ascending=[False,True],kind='mergesort').itemid.astype(int).tolist())
    return seen,seeds,pop

models={};prepared={}
for split in ['val','test']:
    t0=time.perf_counter()
    h=inputs[split]['history'];users=inputs[split]['users']
    models[split]=fit_itemcf(h)
    prepared[split]=prepare_users(h,users)
    print(split,'Item-CF neighbors:',len(models[split]),'fit sec:',round(time.perf_counter()-t0,2))

val Item-CF neighbors: 121194 fit sec: 4.22
test Item-CF neighbors: 139393 fit sec: 5.46


## 3. Generate independent Item-CF, Popularity, and exported FAISS+fallback lists

In [3]:
def itemcf_unpadded(seeds,seen,neighbors,limit=MAX_K):
    scores=defaultdict(float)
    for seed in seeds:
        for item,similarity in neighbors.get(int(seed),[]):
            if item not in seen:scores[item]+=similarity
    return sorted(scores,key=lambda item:(-scores[item],item))[:limit]

def fill_popularity(items,seen,pop,k=MAX_K):
    out=[];used=set()
    for item in items:
        item=int(item)
        if item not in seen and item not in used:
            out.append(item);used.add(item)
            if len(out)==k:return out
    for item in pop:
        if item not in seen and item not in used:
            out.append(item);used.add(item)
            if len(out)==k:return out
    raise RuntimeError('Insufficient unseen catalog items')

sources={};single={};seen_maps={}
for split in ['val','test']:
    users=inputs[split]['users'];seen,seeds,pop=prepared[split];seen_maps[split]=seen
    f=inputs[split]['faiss_rows'].sort_values(['visitorid','rank'],kind='mergesort')
    faiss_lists=f.groupby('visitorid',sort=False).itemid.agg(lambda x:list(map(int,x))).to_dict()
    src={};baseline={'itemcf':{},'faiss_fallback':{},'popularity':{}}
    t0=time.perf_counter()
    for u in users:
        s=seen.get(u,set())
        cf=itemcf_unpadded(seeds.get(u,[]),s,models[split])
        popular=fill_popularity([],s,pop)
        vector=faiss_lists[u]
        assert len(vector)==MAX_K and not (set(vector)&s)
        src[u]={'itemcf':cf,'faiss_fallback':vector,'popularity':popular}
        baseline['itemcf'][u]=fill_popularity(cf,s,pop)
        baseline['faiss_fallback'][u]=vector
        baseline['popularity'][u]=popular
    sources[split]=src;single[split]=baseline
    print(split,'sources built for',len(users),'users in',round(time.perf_counter()-t0,2),'seconds')

val sources built for 13329 users in 1.14 seconds
test sources built for 15137 users in 1.62 seconds


## 4. Weighted Reciprocal Rank Fusion (RRF)

RRF score = Σ source_weight / (60 + rank). Source scores are *not* directly comparable, so fuse ranks. We deliberately keep Item-CF **without fallback** inside fusion; the FAISS export unfortunately already includes fallback, which we explicitly disclose.

In [4]:
def hybrid_rrf(source_lists,seen,pop,weights=SOURCE_WEIGHTS,k=MAX_K,c=RRF_C):
    scores=defaultdict(float)
    source_count=defaultdict(int)
    for name,items in source_lists.items():
        weight=weights[name]
        if weight<=0:continue
        for rank,item in enumerate(items,start=1):
            if item in seen:continue
            scores[item]+=weight/(c+rank)
            source_count[item]+=1
    ranked=sorted(scores,key=lambda item:(-scores[item],-source_count[item],item))
    return fill_popularity(ranked,seen,pop,k)

hybrid={};hybrid_seconds={}
for split in ['val','test']:
    seen,seeds,pop=prepared[split]
    t0=time.perf_counter()
    hybrid[split]={u:hybrid_rrf(sources[split][u],seen.get(u,set()),pop)
                   for u in inputs[split]['users']}
    hybrid_seconds[split]=time.perf_counter()-t0
    assert all(len(v)==MAX_K and len(set(v))==MAX_K and not (set(v)&seen.get(u,set()))
               for u,v in hybrid[split].items())
    print(split,'hybrid fusion seconds (excluding source generation):',round(hybrid_seconds[split],3))

val hybrid fusion seconds (excluding source generation): 2.059
test hybrid fusion seconds (excluding source generation): 2.289


## 5. Evaluate with exactly the same strict denominator as Notebooks 04–07

In [5]:
def evaluate(recs,labels,history_items,seen,k_values=K_VALUES):
    truth=labels.groupby('visitorid').itemid.agg(lambda x:set(map(int,x))).to_dict()
    eligible=sum(int(i in history_items and i not in seen.get(int(u),set()))
                 for u,i in labels[['visitorid','itemid']].itertuples(index=False,name=None))
    rows=[]
    for k in k_values:
        found=0;total=0;per_user=[];hits=[];recommended=set()
        for u,positive in truth.items():
            picks=set(recs[int(u)][:k]);n=len(picks&positive)
            found+=n;total+=len(positive)
            per_user.append(n/len(positive));hits.append(float(n>0));recommended.update(picks)
        rows.append({'K':k,'Recall@K_macro':float(np.mean(per_user)),
                     'Recall@K_micro':found/total,'HitRate@K':float(np.mean(hits)),
                     'UniqueRecommendedItems':len(recommended),
                     'CatalogCoverage':len(recommended)/len(history_items),
                     'EligibleOnlyRecall@K_micro':found/eligible if eligible else np.nan})
    return pd.DataFrame(rows),eligible

metrics={};eligibles={}
for split in ['val','test']:
    h=inputs[split]['history'];labels=inputs[split]['labels']
    catalog=set(map(int,h.itemid.unique()));seen=seen_maps[split]
    metrics[split]={}
    for name,recs in {**single[split],'hybrid_rrf':hybrid[split]}.items():
        m,eligible=evaluate(recs,labels,catalog,seen)
        metrics[split][name]=m;eligibles[split]=eligible
    table=pd.concat([m.assign(method=name) for name,m in metrics[split].items()],ignore_index=True)
    print('\n',split.upper(),'eligible:',eligible,'/',len(labels))
    display(table.loc[table.K.eq(100),['method','Recall@K_micro','HitRate@K','CatalogCoverage','EligibleOnlyRecall@K_micro']].round(5))


 VAL eligible: 21984 / 28634


,method,Recall@K_micro,HitRate@K,CatalogCoverage,EligibleOnlyRecall@K_micro
3,itemcf,0.06237,0.09603,0.40591,0.08124
7,faiss_fallback,0.05141,0.07667,0.20992,0.06696
11,popularity,0.01764,0.03023,0.00082,0.02297
15,hybrid_rrf,0.06594,0.09836,0.41759,0.08588



 TEST eligible: 24875 / 31876


,method,Recall@K_micro,HitRate@K,CatalogCoverage,EligibleOnlyRecall@K_micro
3,itemcf,0.05462,0.08859,0.42600,0.06999
7,faiss_fallback,0.04470,0.06930,0.22619,0.05729
11,popularity,0.01741,0.03131,0.00083,0.02231
15,hybrid_rrf,0.05738,0.08886,0.43833,0.07353


## 6. Candidate-source complementarity

Report overlap and whether a method contributes *unique hits* beyond other retrieval methods; this explains why a hybrid can be useful even if its single-source Recall is lower.

In [6]:
for split in ['val','test']:
    users=inputs[split]['users'];overlaps=[]
    for u in users:
        a=set(single[split]['itemcf'][u]);b=set(single[split]['faiss_fallback'][u])
        overlaps.append(len(a&b)/MAX_K)
    print(split,'mean Item-CF vs FAISS+fallback Top-100 overlap:',round(float(np.mean(overlaps)),4))
    labels=inputs[split]['labels'].groupby('visitorid').itemid.agg(set).to_dict()
    only_cf=only_vec=both=0
    for u,truth in labels.items():
        cf=set(single[split]['itemcf'][int(u)])&truth
        vec=set(single[split]['faiss_fallback'][int(u)])&truth
        only_cf+=len(cf-vec);only_vec+=len(vec-cf);both+=len(cf&vec)
    print('positive pairs: only Item-CF',only_cf,'only FAISS+fallback',only_vec,'both',both)

val mean Item-CF vs FAISS+fallback Top-100 overlap: 0.2943
positive pairs: only Item-CF 854 only FAISS+fallback 540 both 932
test mean Item-CF vs FAISS+fallback Top-100 overlap: 0.2768
positive pairs: only Item-CF 847 only FAISS+fallback 531 both 894


## 7. Export candidates and metrics for Notebook 09

**Interpretation:** If hybrid underperforms Item-CF, keep Item-CF as the baseline and tune fusion on validation only. Do not change fusion weights based on test results. The output includes source membership indicators useful for a future pre-ranking model. Note that FAISS membership includes fallback items.

In [7]:
out_rows=[];summary={
 'method':'weighted_rrf_itemcf_unpadded_faiss_with_fallback_popularity',
 'protocol':'warm-user future-view pairs; exclude seen; cold/seen labels remain in denominator',
 'config':{'rrf_c':RRF_C,'source_weights':SOURCE_WEIGHTS,'max_k':MAX_K,
           'faiss_source_includes_popularity_fallback':True},
 'cutoffs_utc':{'validation':'2015-08-01','test':'2015-09-01'}}
for split,full_name in [('val','validation'),('test','test')]:
    rows=[]
    for u in inputs[split]['users']:
        memberships={name:set(items) for name,items in sources[split][u].items()}
        for rank,item in enumerate(hybrid[split][u],start=1):
            rows.append((u,rank,item,int(item in memberships['itemcf']),
                         int(item in memberships['faiss_fallback']),
                         int(item in memberships['popularity'])))
    path=PROCESSED/f'notebook08_{split}_hybrid_candidates.csv'
    pd.DataFrame(rows,columns=['visitorid','rank','itemid','from_itemcf',
                                'from_faiss_fallback','from_popularity']).to_csv(path,index=False)
    print('Saved:',path,'rows:',len(rows))
    for name,m in metrics[split].items():
        m.to_csv(REPORTS/f'notebook08_{split}_{name}_metrics.csv',index=False)
    summary[full_name]={'users':len(inputs[split]['users']),
                        'positive_pairs':len(inputs[split]['labels']),
                        'eligible_pairs':eligibles[split],
                        'hybrid_fusion_seconds_excluding_source_generation':hybrid_seconds[split],
                        'methods':{name:m.to_dict(orient='records') for name,m in metrics[split].items()}}
json_path=REPORTS/'notebook08_hybrid_retrieval.json'
json_path.write_text(json.dumps(summary,indent=2),encoding='utf-8')
assert eligibles['val']==21984 and eligibles['test']==24875,'Eligibility changed; inspect temporal/evaluation protocol'
assert all(metrics[s]['hybrid_rrf']['Recall@K_micro'].between(0,1).all() for s in ['val','test'])
print('Saved:',json_path)
print('PASS: temporal cutoffs, seen filtering, consistent denominator, hybrid exports')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\processed\notebook08_val_hybrid_candidates.csv rows: 1332900
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\processed\notebook08_test_hybrid_candidates.csv rows: 1513700
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook08_hybrid_retrieval.json
PASS: temporal cutoffs, seen filtering, consistent denominator, hybrid exports
